In [11]:
from datasets import load_dataset

In [12]:
dataset = load_dataset('imdb')
small_train = dataset['train'].shuffle(seed=42).select([i for i in list(range(1000))])
small_test = dataset['test'].shuffle(seed=42).select([i for i in list(range(300))])

In [13]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained('distilbert-base-uncased')

def preprocess_function(examples):
    return tokenizer(examples['text'], truncation=True, padding='max_length', max_length=256)

tokenized_train = small_train.map(preprocess_function, batched=True)
tokenized_test = small_train.map(preprocess_function, batched=True)


In [14]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained('distilbert-base-uncased', num_labels=2)

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 169.62it/s, Materializing param=distilbert.transformer.layer.5.sa_layer_norm.weight]   
DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [17]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir = './results',
    num_train_epochs = 3,
    per_device_train_batch_size = 16,
    per_device_eval_batch_size = 16,
    logging_dir = './logs',
    logging_steps = 10,
    report_to = 'none'
)

trainer = Trainer(
    model = model, 
    args = training_args,
    train_dataset = tokenized_train,
    eval_dataset = tokenized_test
)

trainer.train()

ImportError: Using the `Trainer` with `PyTorch` requires `accelerate>=1.1.0`: Please run `pip install transformers[torch]` or `pip install 'accelerate>=1.1.0'`